# Training YOLO12 PVC

Notebook ini menyiapkan dataset YOLO dari `data/ecg-data`, melatih tiga model YOLO12 detection (`nano`, `small`, `medium`), lalu melakukan inference ke hasil potongan siklus ECG pasien valid di `data/patient/valid/*/ecg/yolo-cycle-input`.

Output inference akan menjawab beat mana yang terdeteksi `Normal` atau `PVC`, menyimpan CSV prediksi, dan membuat visualisasi bbox di atas gambar siklus original yang di-resize/letterbox ke ukuran input YOLO.

## 1. Import library dan konfigurasi

Gunakan `RUN_TRAINING = True` jika ingin mulai training. Training tiga model bisa lama; default dibuat `False` agar notebook aman dijalankan untuk inspeksi awal.

In [9]:
from __future__ import annotations

import json
import random
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook-preprocess" else Path.cwd()
SOURCE_DATA_DIR = PROJECT_ROOT / "data" / "ecg-data"
SOURCE_DATA_YAML = SOURCE_DATA_DIR / "data.yaml"
DETECT_DATA_DIR = PROJECT_ROOT / "data" / "ecg-data-detect"
DETECT_DATA_YAML = DETECT_DATA_DIR / "data.yaml"
VALID_PATIENT_DIR = PROJECT_ROOT / "data" / "patient" / "valid"
CYCLE_CSV_PATH = VALID_PATIENT_DIR / "ecg_12lead_cycles_yolo.csv"
RUNS_DIR = PROJECT_ROOT / "runs" / "yolo_pvc"

YOLO12_MODELS = {
    "nano": "yolo12n.pt",
    "small": "yolo12s.pt",
    "medium": "yolo12m.pt",
}

RUN_TRAINING = True
RUN_INFERENCE = True
TRAIN_EPOCHS = 100
TRAIN_IMGSZ = 640
TRAIN_BATCH = 16
TRAIN_PATIENCE = 25
TRAIN_SEED = 42
TRAIN_WORKERS = 4
INFERENCE_CONF = 0.25
INFERENCE_IOU = 0.50
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Project root      : {PROJECT_ROOT}")
print(f"Dataset source    : {SOURCE_DATA_DIR}")
print(f"Detection dataset : {DETECT_DATA_DIR}")
print(f"Device            : {DEVICE}")
print(f"CUDA available    : {torch.cuda.is_available()}")

Project root      : /home/nugee/code-program/code-thesis/thesis-severity
Dataset source    : /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data
Detection dataset : /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-detect
Device            : 0
CUDA available    : True


## 2. Validasi dataset dan konversi label ke detection bbox

Dataset Roboflow yang ada dapat berisi polygon segmentation. Untuk YOLO12 detection, label dikonversi ke bbox YOLO: `class x_center y_center width height`. Jika file label sudah bbox 5 kolom, formatnya dipertahankan.

In [10]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = ["Normal", "PVC"]


def load_source_yaml() -> dict:
    if not SOURCE_DATA_YAML.exists():
        raise FileNotFoundError(f"YAML dataset tidak ditemukan: {SOURCE_DATA_YAML}")
    return yaml.safe_load(SOURCE_DATA_YAML.read_text())


def list_source_pairs() -> list[tuple[Path, Path]]:
    image_dir = SOURCE_DATA_DIR / "train" / "images"
    label_dir = SOURCE_DATA_DIR / "train" / "labels"
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError("Folder train/images atau train/labels tidak ditemukan di data/ecg-data")

    pairs = []
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label_path = label_dir / f"{image_path.stem}.txt"
        if label_path.exists():
            pairs.append((image_path, label_path))
    return pairs


def clamp01(value: float) -> float:
    return max(0.0, min(1.0, float(value)))


def convert_yolo_label_line_to_bbox(line: str) -> str | None:
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    class_id = int(float(parts[0]))
    values = [float(value) for value in parts[1:]]

    if len(values) == 4:
        x_center, y_center, width, height = values
    elif len(values) >= 6 and len(values) % 2 == 0:
        xs = values[0::2]
        ys = values[1::2]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_center = (x_min + x_max) / 2
        y_center = (y_min + y_max) / 2
        width = x_max - x_min
        height = y_max - y_min
    else:
        return None

    x_center = clamp01(x_center)
    y_center = clamp01(y_center)
    width = clamp01(width)
    height = clamp01(height)
    if width <= 0 or height <= 0:
        return None

    return f"{class_id} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}"


def convert_label_file_to_detection(source_label: Path, destination_label: Path) -> int:
    converted_lines = []
    for line in source_label.read_text().splitlines():
        converted = convert_yolo_label_line_to_bbox(line)
        if converted:
            converted_lines.append(converted)
    destination_label.write_text("\n".join(converted_lines) + ("\n" if converted_lines else ""))
    return len(converted_lines)


source_yaml = load_source_yaml()
source_pairs = list_source_pairs()
print(f"YAML names : {source_yaml.get('names')}")
print(f"Image-label pairs: {len(source_pairs)}")
print("Contoh pair:", source_pairs[0] if source_pairs else None)

YAML names : ['Normal', 'PVC']
Image-label pairs: 1990
Contoh pair: (PosixPath('/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data/train/images/0-2-_jpg.rf.QcKt3XQ2QmDnPOqJTvMg.jpg'), PosixPath('/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data/train/labels/0-2-_jpg.rf.QcKt3XQ2QmDnPOqJTvMg.txt'))


## 3. Buat split dataset detection lokal

Jika folder valid/test belum tersedia, notebook membuat split deterministic dari data train: 80% train, 10% valid, 10% test. Hasilnya disimpan di `data/ecg-data-detect/` agar dataset asli tidak berubah.

In [11]:
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}


def reset_detection_dataset() -> None:
    if DETECT_DATA_DIR.exists():
        shutil.rmtree(DETECT_DATA_DIR)
    for split in SPLIT_RATIOS:
        (DETECT_DATA_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (DETECT_DATA_DIR / split / "labels").mkdir(parents=True, exist_ok=True)


def split_pairs(pairs: list[tuple[Path, Path]]) -> dict[str, list[tuple[Path, Path]]]:
    rng = random.Random(TRAIN_SEED)
    shuffled = pairs.copy()
    rng.shuffle(shuffled)

    n_total = len(shuffled)
    n_train = int(n_total * SPLIT_RATIOS["train"])
    n_val = int(n_total * SPLIT_RATIOS["val"])
    return {
        "train": shuffled[:n_train],
        "val": shuffled[n_train:n_train + n_val],
        "test": shuffled[n_train + n_val:],
    }


def prepare_detection_dataset() -> pd.DataFrame:
    reset_detection_dataset()
    splits = split_pairs(source_pairs)
    records = []

    for split_name, split_pairs_ in splits.items():
        image_out_dir = DETECT_DATA_DIR / split_name / "images"
        label_out_dir = DETECT_DATA_DIR / split_name / "labels"
        for image_path, label_path in split_pairs_:
            image_out = image_out_dir / image_path.name
            label_out = label_out_dir / f"{image_path.stem}.txt"
            shutil.copy2(image_path, image_out)
            object_count = convert_label_file_to_detection(label_path, label_out)
            records.append({
                "split": split_name,
                "image_path": image_out,
                "label_path": label_out,
                "object_count": object_count,
            })

    data_yaml = {
        "path": str(DETECT_DATA_DIR.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 2,
        "names": CLASS_NAMES,
    }
    DETECT_DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
    return pd.DataFrame(records)


detect_dataset_df = prepare_detection_dataset()
print(f"Detection YAML: {DETECT_DATA_YAML}")
display(detect_dataset_df.groupby("split").agg(images=("image_path", "count"), objects=("object_count", "sum")))
print(DETECT_DATA_YAML.read_text())

Detection YAML: /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-detect/data.yaml


,images,objects
split,,
test,199,199
train,1592,1596
val,199,199


path: /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-detect
train: train/images
val: val/images
test: test/images
nc: 2
names:
- Normal
- PVC



## 4. Training YOLO12 nano, small, medium

Ultralytics YOLO12 detection weights yang digunakan: `yolo12n.pt`, `yolo12s.pt`, dan `yolo12m.pt`. Set `RUN_TRAINING = True` di cell konfigurasi untuk menjalankan training.

In [12]:
def model_run_dir(model_variant: str) -> Path:
    return RUNS_DIR / f"yolo12_{model_variant}_pvc"


def best_weight_path(model_variant: str) -> Path:
    return model_run_dir(model_variant) / "weights" / "best.pt"


def train_single_model(model_variant: str, base_weight: str):
    print(f"\n=== Training YOLO12 {model_variant}: {base_weight} ===")
    model = YOLO(base_weight)
    results = model.train(
        data=str(DETECT_DATA_YAML),
        epochs=TRAIN_EPOCHS,
        imgsz=TRAIN_IMGSZ,
        batch=TRAIN_BATCH,
        patience=TRAIN_PATIENCE,
        device=DEVICE,
        workers=TRAIN_WORKERS,
        seed=TRAIN_SEED,
        project=str(RUNS_DIR),
        name=f"yolo12_{model_variant}_pvc",
        exist_ok=True,
        task="detect",
    )
    return results


def train_all_models() -> dict[str, Path]:
    trained_weights = {}
    if not RUN_TRAINING:
        print("RUN_TRAINING=False, training dilewati. Set True untuk melatih model.")
        for variant in YOLO12_MODELS:
            weight_path = best_weight_path(variant)
            if weight_path.exists():
                trained_weights[variant] = weight_path
        return trained_weights

    for variant, base_weight in YOLO12_MODELS.items():
        train_single_model(variant, base_weight)
        trained_weights[variant] = best_weight_path(variant)
    return trained_weights


trained_weight_paths = train_all_models()
trained_weight_paths


=== Training YOLO12 nano: yolo12n.pt ===
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11903MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-detect/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

{'nano': PosixPath('/home/nugee/code-program/code-thesis/thesis-severity/runs/yolo_pvc/yolo12_nano_pvc/weights/best.pt'),
 'small': PosixPath('/home/nugee/code-program/code-thesis/thesis-severity/runs/yolo_pvc/yolo12_small_pvc/weights/best.pt'),
 'medium': PosixPath('/home/nugee/code-program/code-thesis/thesis-severity/runs/yolo_pvc/yolo12_medium_pvc/weights/best.pt')}

## 5. Validasi model terlatih

Cell ini menjalankan validasi untuk model yang sudah punya `best.pt`. Jika training belum dijalankan, cell akan skip.

In [13]:
def validate_trained_models(weight_paths: dict[str, Path]) -> pd.DataFrame:
    records = []
    for variant, weight_path in weight_paths.items():
        if not weight_path.exists():
            print(f"[SKIP] {variant}: weight belum ada ({weight_path})")
            continue
        model = YOLO(str(weight_path))
        metrics = model.val(data=str(DETECT_DATA_YAML), imgsz=TRAIN_IMGSZ, device=DEVICE, split="val")
        records.append({
            "variant": variant,
            "weight_path": weight_path,
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
        })
    return pd.DataFrame(records)


validation_df = validate_trained_models(trained_weight_paths)
display(validation_df)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11903MiB)
YOLOv12n summary (fused): 159 layers, 2,557,118 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 652.1±317.5 MB/s, size: 7.2 KB)
val: Scanning /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-detect/val/labels.cache... 199 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 199/199 119.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 8.8it/s 1.5s0.1s
                   all        199        199      0.992          1      0.995      0.992
                Normal        132        132          1          1      0.995      0.994
                   PVC         67         67      0.984          1      0.995      0.991
Speed: 1.0ms preprocess, 4.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/nugee/code-program/code-thesis/thesis-severit

,variant,weight_path,map50,map50_95
0,nano,/home/nugee/code-program/code-thesis/thesis-se...,0.995,0.992178
1,small,/home/nugee/code-program/code-thesis/thesis-se...,0.995,0.987111
2,medium,/home/nugee/code-program/code-thesis/thesis-se...,0.995,0.992869
